a notebook to test Tensorflow

In [2]:
import tensorflow as tf
print('TensorFlow version:', tf.__version__)

2026-02-08 20:47:35.171973: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-08 20:47:35.482994: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-08 20:47:35.546687: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-02-08 20:47:35.546705: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if yo

TensorFlow version: 2.10.0


In [3]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data("mnist.npz")

In [5]:
x_train.data, x_train.shape, x_train.dtype

(<memory at 0x7f62803867c0>, (60000, 28, 28), dtype('uint8'))

In [6]:
x_train.min(), x_train.max()

(0, 255)

In [7]:
# Since many algorithms and models are sensitive to the scale of the features,
# we often center and scale features into a range such as [0, 1] or [-1, 1].
# In our case, we can do this easily by dividing the images by 255.
def preprocess(dataset):
    return dataset/255.0

In [8]:
x_train = preprocess(x_train)
x_test = preprocess(x_test)

In [ ]:
# let's check these properties
x_train.dtype, x_train.min(), x_train.max()

(dtype('float64'), 0.0, 1.0)

### Instantiate a simple multilayer neural network model

- First, we use Flatten to expand the two-dimensional images into a one dimensional array by specifying the input shape as 28 × 28.
- The second layer is densely connected and uses the 'relu' activation function to introduce some non linearity.
- The third layer is a dropout layer to reduce overfitting and make the model more generalizable.

Since the handwritten digits consist of 10 different digits from 0 to 9, our last layer is densely connected for 10-class classification with `softmax` activation.

In [12]:
# The sequential model definition
model = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(input_shape=(28, 28)),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dropout(0.2),
  tf.keras.layers.Dense(10, activation='softmax')
])

2026-02-08 21:27:41.872757: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:966] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-02-08 21:27:41.876425: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-02-08 21:27:41.877512: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory
2026-02-08 21:27:41.878416: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublasLt.so.11'; dlerror: libcublasLt.so.11: cannot open shared object file: No such file or directory
2026-02-08 21:27:41.879406: W tensorflow/stream_executor/platform/default/dso_loader.cc:6

After we’ve defined the model architecture, we need to specify three different components:
- the evaluation metric
- loss function
- optimizer

In [13]:
# Model compilation with optimizer, loss function, and optimizer
model.compile(optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

In [14]:
# start our model training with five epochs
model.fit(x_train, y_train, epochs=5)

Epoch 1/5
1875/1875 [==============================] - 4s 1ms/step - loss: 0.2998 - accuracy: 0.9125
Epoch 2/5
1875/1875 [==============================] - -0s -141us/step - loss: 0.1473 - accuracy: 0.9561
Epoch 3/5
1875/1875 [==============================] - 2s 1ms/step - loss: 0.1108 - accuracy: 0.9668
Epoch 4/5
1875/1875 [==============================] - 2s 1ms/step - loss: 0.0900 - accuracy: 0.9720
Epoch 5/5
1875/1875 [==============================] - 2s 1ms/step - loss: 0.0770 - accuracy: 0.9760


In [15]:
model.evaluate(x_test, y_test)

313/313 [==============================] - 0s 872us/step - loss: 0.0766 - accuracy: 0.9771


[0.07662195712327957, 0.9771000146865845]

After we’ve trained the model and are happy with its performance, we can save it so that we don’t have to retrain it from scratch next time.

This code saves the model as file `my_model.h5` in the current working directory. When
we start a new Python session, we can import TensorFlow and load the model object
from the `my_model.h5` file using
```py
import tensorflow as tf
model = tf.keras.models.load_model('my_model.h5')
```

if you want to save the python environment and be able to create it again
```bash
# Export the entire conda environment including conda and pip packages
conda env export > environment.yml

# To recreate the environment later:
conda env create -f environment.yml
```

In [16]:
model.save('my_model.h5')

In [ ]:
from tensorflow import keras
import keras_tuner as kt

We trained a model using TensorFlow’s Keras API for a single set of
hyperparameters. These hyperparameters remain constant over the training process
and directly affect the performance of your machine learning program. Let’s learn
how to tune hyperparameters for your TensorFlow program with Keras Tuner

In [ ]:
# Define a model-building function that takes an argument of type HyperParameters and returns a compiled Keras Model.
# The HyperParameters object allows you to define the search space for the hyperparameters of your model.
# In this example, we are tuning the number of units in the first Dense layer and the learning rate for the Adam optimizer.
def model_builder(hp):
  model = keras.Sequential()
  model.add(keras.layers.Flatten(input_shape=(28, 28)))

  hp_units = hp.Int('units', min_value=32, max_value=512, step=32)
  model.add(keras.layers.Dense(units=hp_units, activation='relu'))
  model.add(keras.layers.Dense(10))

  hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
  model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate), 
                loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), 
                metrics=['accuracy'])
  return model